In [ ]:
# %pip install folium
# %pip install geopy
# %pip install geopy

In [ ]:
import pandas as pd
import folium
import time
from geopy.geocoders import Nominatim

```txt
실습 목표
  여러 아파트의 주소를 반복적으로 지오코딩할 수 있다.
  지오코딩 실패를 필터링하여 지도 오류를 방지할 수 있다.
  외부 서비스 요청 제한을 고려해 time.sleep()으로 요청 딜레이를 적용할 수 있다.
  folium으로 여러 마커를 지도에 동시에 표시하고, 툴팁/팝업을 구성할 수 있다.
  결과를 html로 저장하여 브라우저에서 확인할 수 있다.

실습 과제

과제 A(기본)
  요청 제한을 고려해 30개만 골라서 지도에 마커를 출력하기.
  실패한 주소는 제외하고 성공한 건수/실패 건수를 출력하기.

과제 B
  거래 금액이 **상위 20%** 인 데이터만 지도에 표시하기.

과제 C
  같은 법정동 주소가 반복되면 중복 마커를 1개로 출력하기.
```

STEP#1 - 데이터 로그 + 주소 컬럼 만들기

In [ ]:
# lat = [[]]
# lng = [[]]
# 지오코딩 입력용 주소 만들기 ( 형식 맞출 것 )
df = pd.read_csv('../../data/trade_apt_api_2023_address.txt', sep="\t")
df

In [ ]:
df['주소'] = '서울시 '+df['지역명']+' '+df['법정동']+' '+df['지번']
print(df['주소'])

In [ ]:
df['주소'].head(3)

In [ ]:
df['주소'].tail(3)

STEP#2 - 지오코딩 준비

In [ ]:
geo_local = Nominatim(user_agent='apt-map-trainer')
def geocoding(address):
  try:
    geo = geo_local.geocode(address)
    if geo is None:
      return None, None
    return geo.latitude, geo.longitude
  except:
    return None, None

STEP#3 - 지도에 찍을 대상 선정( 실습 포인트 )

In [ ]:
#N = 10
N = 3
sample = df.head( N ).copy()
print(sample)

STEP#4 - 여러 건 지오코딩 + 실패 필터링 + 지연처리

In [ ]:
success_rows = []
fail_count = 0

for i, row in sample.iterrows():
  address = row['주소']

  time.sleep(1)
  lat, lng = geocoding(address)
  if lat is None or lng is None:
    fail_count += 1
    continue # 실패한 건은 지도에서 제외하기 - 조건문으로 이동함

  # 성공한 데이터만 모아두기( 나중에 지도에 출력할 때 사용함 )
  success_rows.append((lat, lng, row))
print(f'성공: {len(success_rows)} 건/ 실패: {fail_count} 건')

STEP#5 - 지도 생성

In [ ]:
if len(success_rows) == 0:
  print('네트워크를 확인하세요')
  # raise ValueError('지오코딩 성공 데이터가 0건 입니다. 주소 형식/네트워크를 확인하세요')
# center_lat, center_lng, _ = success_rows[0]

lat = 36.763718
lng = 127.28194050
m = folium.Map(location=[lat, lng], zoom_start = 15, titles='OpenStreetMap') 

STEP#6 - 여러 마커 추가

In [ ]:
for lat, lng, row in success_rows:
  price = row['거래금액']
  dong = row['법정동']
  ym = row['기준년월']

  # tooltip: 마커에 마우스 올렸을 때
  tooltip = f'{dong} / {price:,}만원'

  # popup: 클릭했을 때 상세정보( html 기능 )
  popup = folium.Popup(
    f'''
      <b>기준년월</b> : {ym}<br/>
      <b>법정동</b> : {dong}<br/>
      <b>거래금액</b> : {price}<br/>
      <b>주소</b> : {row["주소"]}<br/>
    ''',
    max_width=300
  )
  folium.Marker(
    location=[lat, lng],
    tooltip=tooltip,
    popup=popup
  ).add_to(m)

STEP#7 - 결과확인 + 저장

In [ ]:
m.save('아파트-멀티마커.html')
m